In [1]:
import pandas as pd
import json
import re
import numpy as np


In [12]:

df = pd.read_pickle("data_250924.pkl")

In [4]:
df

,publish year,doi,title,document type,keywords,authors with affiliations,paper id,pv technology (raw text),pv tech,pv technology detail,...,latitude,longitude,koppen_zone,Main climate,Precipitation,Temperature,PV zone,source of value,faults_list,avg_year
1,2018.0,10.7567/JJAP.57.08RG07,Long-term performance and degradation rate ana...,Conference paper,NaN,"Bayandelger B.-E., Department of Electrical En...",1,single-crystalline silicon (sc-Si),mono-c-Si,NaN,...,44.9000,110.1200,BWk,B,W,k,Desert,text,"[lid, short-circuit current loss]",2007.0
2,2018.0,10.7567/JJAP.57.08RG07,Long-term performance and degradation rate ana...,Conference paper,NaN,"Bayandelger B.-E., Department of Electrical En...",1,multicrystalline silicon (mc-Si),multi-c-Si,NaN,...,44.9000,110.1200,BWk,B,W,k,Desert,text,[short-circuit current loss],2007.0
4,2015.0,10.1016/j.solener.2015.04.030,Novel method for the improvement in the evalua...,Article,Degradation; Initial degradation; Performance ...,"Belluardo G., Institute for Renewable Energy, ...",3,mono-crystalline silicon (mc-Si),multi-c-Si,NaN,...,46.4600,11.3300,Cfb,C,f,b,Moderate,,[],2012.0
5,2015.0,10.1016/j.solener.2015.04.030,Novel method for the improvement in the evalua...,Article,Degradation; Initial degradation; Performance ...,"Belluardo G., Institute for Renewable Energy, ...",3,poly-crystalline silicon (pc-Si),multi-c-Si,NaN,...,46.4600,11.3300,Cfb,C,f,b,Moderate,,[],2012.0
6,2015.0,10.1016/j.solener.2015.04.030,Novel method for the improvement in the evalua...,Article,Degradation; Initial degradation; Performance ...,"Belluardo G., Institute for Renewable Energy, ...",3,ribbon poly-crystalline silicon,multi-c-Si,ribbon technology,...,46.4600,11.3300,Cfb,C,f,b,Moderate,table,[],2012.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5748,2019.0,10.1016/j.nanoen.2018.12.072,A leaf-inspired photon management scheme using...,article,All dielectric; Graphene Si Schottky junction ...,"Das S., NanoScience Technology Center, Univers...",19775,graphene/Si Schottky junction solar cell,other,"bilayer graphene electrode, ultra-thin 20 µm S...",...,28.5383,-81.3792,Cfa,C,f,a,Hot & Humid,text,[],NaN
5753,2023.0,10.1016/j.jobe.2023.107947,A holistic assessment of the photovoltaic-ener...,Article,Diffusion of innovations theory; Economic and ...,"Chen X., School of Urban Design, Wuhan Univers...",19816,monocrystalline silicon PV modules,mono-c-Si,NaN,...,30.5966,114.3853,Cfa,C,f,a,Hot & Humid,text,[shading],2033.5
5784,2024.0,10.1109/ACCESS.2024.3369914,An Evaluation of Battery Degradation and Predi...,Article,average approach; Battery lifetime; global bat...,"Kumba K., Vellore Institute of Technology, Tam...",19978,solar photovoltaics system,NaN,NaN,...,13.0827,80.2707,Aw,A,w,NaN,Hot & Humid,text,[not reported],2022.0
5786,2024.0,10.1016/j.solmat.2023.112655,Highly efficient double-side-passivated perovs...,Article,NaN,"Dipta S.S., School of Photovoltaic and Renewab...",19986,Hybrid metal halide perovskite solar cells (PS...,other,"n-i-p structured PSC, n-i-p structured PSCs",...,-33.8688,151.2093,Cfa,C,f,a,Hot & Humid,text,"[ion migration, interface defects]",NaN


In [13]:
FAULTS_COL = 'faults_list'

reported_faults_mask = df[FAULTS_COL].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    )

not_reported_faults_mask = df[FAULTS_COL].apply(
    lambda x: isinstance(x, list) and len(x) == 0
)

In [14]:
faults_filter = ['reported', 'not_reported']
faults_mask = False

In [15]:
if 'reported' in faults_filter:
        faults_mask |= reported_faults_mask

if 'not_reported' in faults_filter:
    faults_mask |= not_reported_faults_mask

df = df[faults_mask]
df

,publish year,doi,title,document type,keywords,authors with affiliations,paper id,pv technology (raw text),pv tech,pv technology detail,...,latitude,longitude,koppen_zone,Main climate,Precipitation,Temperature,PV zone,source of value,faults_list,avg_year
1,2018.0,10.7567/JJAP.57.08RG07,Long-term performance and degradation rate ana...,Conference paper,NaN,"Bayandelger B.-E., Department of Electrical En...",1,single-crystalline silicon (sc-Si),mono-c-Si,NaN,...,44.9000,110.1200,BWk,B,W,k,Desert,text,"[lid, short-circuit current loss]",2007.0
2,2018.0,10.7567/JJAP.57.08RG07,Long-term performance and degradation rate ana...,Conference paper,NaN,"Bayandelger B.-E., Department of Electrical En...",1,multicrystalline silicon (mc-Si),multi-c-Si,NaN,...,44.9000,110.1200,BWk,B,W,k,Desert,text,[short-circuit current loss],2007.0
4,2015.0,10.1016/j.solener.2015.04.030,Novel method for the improvement in the evalua...,Article,Degradation; Initial degradation; Performance ...,"Belluardo G., Institute for Renewable Energy, ...",3,mono-crystalline silicon (mc-Si),multi-c-Si,NaN,...,46.4600,11.3300,Cfb,C,f,b,Moderate,,[],2012.0
5,2015.0,10.1016/j.solener.2015.04.030,Novel method for the improvement in the evalua...,Article,Degradation; Initial degradation; Performance ...,"Belluardo G., Institute for Renewable Energy, ...",3,poly-crystalline silicon (pc-Si),multi-c-Si,NaN,...,46.4600,11.3300,Cfb,C,f,b,Moderate,,[],2012.0
6,2015.0,10.1016/j.solener.2015.04.030,Novel method for the improvement in the evalua...,Article,Degradation; Initial degradation; Performance ...,"Belluardo G., Institute for Renewable Energy, ...",3,ribbon poly-crystalline silicon,multi-c-Si,ribbon technology,...,46.4600,11.3300,Cfb,C,f,b,Moderate,table,[],2012.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5748,2019.0,10.1016/j.nanoen.2018.12.072,A leaf-inspired photon management scheme using...,article,All dielectric; Graphene Si Schottky junction ...,"Das S., NanoScience Technology Center, Univers...",19775,graphene/Si Schottky junction solar cell,other,"bilayer graphene electrode, ultra-thin 20 µm S...",...,28.5383,-81.3792,Cfa,C,f,a,Hot & Humid,text,[],NaN
5753,2023.0,10.1016/j.jobe.2023.107947,A holistic assessment of the photovoltaic-ener...,Article,Diffusion of innovations theory; Economic and ...,"Chen X., School of Urban Design, Wuhan Univers...",19816,monocrystalline silicon PV modules,mono-c-Si,NaN,...,30.5966,114.3853,Cfa,C,f,a,Hot & Humid,text,[shading],2033.5
5784,2024.0,10.1109/ACCESS.2024.3369914,An Evaluation of Battery Degradation and Predi...,Article,average approach; Battery lifetime; global bat...,"Kumba K., Vellore Institute of Technology, Tam...",19978,solar photovoltaics system,NaN,NaN,...,13.0827,80.2707,Aw,A,w,NaN,Hot & Humid,text,[not reported],2022.0
5786,2024.0,10.1016/j.solmat.2023.112655,Highly efficient double-side-passivated perovs...,Article,NaN,"Dipta S.S., School of Photovoltaic and Renewab...",19986,Hybrid metal halide perovskite solar cells (PS...,other,"n-i-p structured PSC, n-i-p structured PSCs",...,-33.8688,151.2093,Cfa,C,f,a,Hot & Humid,text,"[ion migration, interface defects]",NaN


In [7]:
reported_faults_mask.sum()

np.int64(1378)

In [3]:
df.columns

Index(['publish year', 'doi', 'title', 'document type', 'keywords',
       'authors with affiliations', 'paper id', 'pv technology (raw text)',
       'pv tech', 'pv technology detail', 'scope of study',
       'duration (raw text)', 'duration in years', 'start year', 'end year',
       'system capacity (raw text)', 'system capacity in watts',
       'number of pv modules in study', 'pv module nominal power (raw text)',
       'pv module nominal power in watts', 'country', 'location (raw text)',
       'location', 'pv module name', 'bifacial', 'mounting',
       'annual power degradation rate (raw text)',
       'annual power degradation rate in percent', 'location of information',
       'confidence level', 'annual degradation rate of other parameters',
       'degradation metric', 'analysis method',
       'number of measurements for degradation analysis',
       'source of initial power value', 'other examination techs',
       'faults (raw text)', 'faults', 'grid-connected',
      

In [ ]:


def normalize_object_columns(df, mode="string"):
    """
    mode = "string" : list -> 'a, b, c'
    mode = "json"   : list/dict -> JSON string (可逆)
    """
    df = df.copy()

    for col in df.columns:
        if df[col].dtype == "object":
            if df[col].apply(lambda x: isinstance(x, (list, dict))).any():

                if mode == "string":
                    df[col] = df[col].apply(
                        lambda x: ", ".join(map(str, x)) if isinstance(x, list)
                        else json.dumps(x) if isinstance(x, dict)
                        else x
                    )

                elif mode == "json":
                    df[col] = df[col].apply(
                        lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x
                    )

    return df

# 2. 处理所有数值列里的文本
numeric_cols = [
    "duration in years",
    "system capacity in watts",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

def normalize_multi_numeric(x):
    """
    从 list / string 中提取所有数值
    - list: [295.0, 300.0]
    - "160.0, 210.0"
    - "A: 295.0"
    - "A: 295.0, B: 300.0"
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan

    # 情况 1：list
    if isinstance(x, list):
        vals = [v for v in x if isinstance(v, (int, float))]
        return np.mean(vals) if vals else np.nan

    # 情况 2：string → 用正则提取所有数字
    if isinstance(x, str):
        numbers = re.findall(r"[-+]?\d*\.?\d+", x)
        numbers = [float(n) for n in numbers]
        return np.mean(numbers) if numbers else np.nan

    # 情况 3：已经是数值
    if isinstance(x, (int, float)):
        return float(x)

    return np.nan

df["pv module nominal power in watts"] = (
    df["pv module nominal power in watts"]
    .apply(normalize_multi_numeric)
)

def normalize_boolean(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return pd.NA

    if isinstance(x, bool):
        return x

    if isinstance(x, (int, float)):
        return bool(x)

    if isinstance(x, str):
        v = x.strip().lower()
        if v in {"true", "yes", "y", "1", "bifacial"}:
            return True
        if v in {"false", "no", "n", "0", "monofacial"}:
            return False
        if v in {"not reported", "unknown", "na", "n/a", ""}:
            return pd.NA

    return pd.NA

boolean_cols = [
    "bifacial",
    "grid-connected",
]

for col in boolean_cols:
    df[col] = df[col].apply(normalize_boolean).astype("boolean")

df = normalize_object_columns(df, mode="string")
df.to_parquet("data_250924.parquet", engine="pyarrow")